In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

In [2]:
df = pd.read_csv('/Users/nikita/Desktop/11. Empirical Industrial Organisation and Market Design/Problem sets/top_paid_apps_details.csv')
df
columns_to_delete = [
    'advisories', 'kind', 'trackContentRating', 'trackCensoredName',
    'fileSizeBytes', 'sellerUrl', 'formattedPrice', 'contentAdvisoryRating',
    'artistId', 'artistName', 'genres', 'primaryGenreId', 'bundleId',
    'releaseDate', 'trackId', 'trackName', 'genreIds', 'sellerName', 'wrapperType'
]

df.drop(columns=columns_to_delete, inplace=True)
total_downloads = df['downloads'].sum()
total_outside = df[df['outside_good'] == 'outside']['downloads'].sum()

# shares
ln_outside_share = np.log(total_outside / total_downloads)
df['ln_shares'] = np.log(df['downloads'] / total_downloads)
df['shares_difference'] = df['ln_shares'] - ln_outside_share

# normalise prices w.r.t. EUR (exchange rate is the annual average exchange rate provided by ECB statistics)
exchange_rates = {
    'EUR': 1,
    'USD': 0.9196,
    'AUD': 0.6094,
    'JPY': 0.006114,
    'SEK': 0.08767,
    'KRW': 0.0006826,
    'GBP': 1.1743
}
df['price_in_EUR'] = df['price'] * df['currency'].map(exchange_rates)

# create a column of dummies
df['in_app_dummy'] = np.where(df['in_app'] == True, 1, 0)

1.1 Berry Logit

In [3]:
# regression without IV
df_filtered = df[df['outside_good'] != 'outside']
X = df_filtered[['price_in_EUR', 'averageUserRating', 'in_app_dummy']]
X = sm.add_constant(X)
y = df_filtered['shares_difference']

model = sm.OLS(y, X)
results = model.fit()

print(results.summary())

with open('logit_no_IV_regression_results.tex', 'w') as f:
    f.write(results.summary().as_latex())
# introduce an instrument, as proposed in the problem set, we will use the number of apps in the considered category

# create a new column 
category_counts = df['primaryGenreName'].value_counts()
df['category_count'] = df['primaryGenreName'].map(category_counts)

# IV-regressions (two stages least squares)
df_filtered = df[df['outside_good'] != 'outside']
Z = df_filtered['category_count'] # IV
X =  df_filtered['price_in_EUR'] # Endogenous variable
X1 = df_filtered['averageUserRating']# Exogenous variable 1
X2 = df_filtered['in_app_dummy']# Exogenous variable 2
y = df_filtered['shares_difference']

data = pd.DataFrame({'shares_difference': y, 'averageUserRating': X1, 'in_app_dummy': X2, 'price_in_EUR' : X, 'apps_in_genre': Z})
data['const'] = 1


model = IV2SLS(
    dependent=data['shares_difference'],                  
    exog=data[['const', 'averageUserRating', 'in_app_dummy']],  
    instruments=data['apps_in_genre'],            
    endog=data['price_in_EUR']       
)

#  model fitting
results = model.fit()
print(results.summary) 

# the first stage results
first_stage_results = results.first_stage
print(first_stage_results.summary)


                            OLS Regression Results                            
Dep. Variable:      shares_difference   R-squared:                       0.020
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     33.63
Date:                Sun, 10 Nov 2024   Prob (F-statistic):           1.63e-21
Time:                        22:48:50   Log-Likelihood:                -10232.
No. Observations:                5069   AIC:                         2.047e+04
Df Residuals:                    5065   BIC:                         2.050e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const               -10.3079      0.27

1.2 Nested Logit

In [4]:
# create the within-group market share of good j
category_total_downloads = df.groupby('primaryGenreName')['downloads'].transform('sum')
df['ln_share_within_group'] = np.log(df['downloads'] / category_total_downloads)

In [5]:
# the regression without IV
df_filtered = df[df['outside_good'] != 'outside']
X = df_filtered[['price_in_EUR', 'averageUserRating', 'in_app_dummy', 'ln_share_within_group']]
X = sm.add_constant(X)
y = df_filtered['shares_difference']

model = sm.OLS(y, X)
results = model.fit()

print(results.summary())


                            OLS Regression Results                            
Dep. Variable:      shares_difference   R-squared:                       0.539
Model:                            OLS   Adj. R-squared:                  0.539
Method:                 Least Squares   F-statistic:                     1480.
Date:                Sun, 10 Nov 2024   Prob (F-statistic):               0.00
Time:                        22:48:50   Log-Likelihood:                -8319.9
No. Observations:                5069   AIC:                         1.665e+04
Df Residuals:                    5064   BIC:                         1.668e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -3.97

In [6]:
# IV-regressions (two stages least squares)
df_filtered = df[df['outside_good'] != 'outside']
Z = df_filtered['category_count'] # IV
X = df_filtered['price_in_EUR'] # Endogenous variable
X1 = df_filtered['averageUserRating']# Exogenous variable 1
X2 = df_filtered['in_app_dummy'] # Exogenous variable 2
X3 =  df_filtered['ln_share_within_group'] # Exogenous variable 3
y = df_filtered['shares_difference']

data = pd.DataFrame({'shares_difference': y, 'averageUserRating': X1, 'in_app_dummy': X2, 'ln_share_within_group' : X3, 'price_in_EUR' : X, 'apps_in_genre': Z})
data['const'] = 1


model = IV2SLS(
    dependent=data['shares_difference'],                  
    exog=data[['const', 'averageUserRating', 'in_app_dummy', 'ln_share_within_group']],  
    instruments=data['apps_in_genre'],            
    endog=data['price_in_EUR']       
)

#  model fitting
results = model.fit()
print(results.summary) 

# the first stage results
first_stage_results = results.first_stage
print(first_stage_results.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:      shares_difference   R-squared:                     -6.0723
Estimator:                    IV-2SLS   Adj. R-squared:                -6.0779
No. Observations:                5069   F-statistic:                    639.54
Date:                Sun, Nov 10 2024   P-value (F-stat)                0.0000
Time:                        22:48:50   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                                   Parameter Estimates                                   
                       Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------
const                    -1.7637     0.6371    -2.7684     0.0056     -3.0123     -0.5150
averageU

2.1 Cost Estimation

In [7]:
# IV-regressions (two stages least squares) using only data for US market
df_filtered = df[(df['outside_good'] != 'outside') & (df['country'] == 'us')]
Z = df_filtered['category_count'] # IV
X =  df_filtered['price_in_EUR'] # Endogenous variable
X1 = df_filtered['averageUserRating']# Exogenous variable 1
X2 = df_filtered['in_app_dummy']# Exogenous variable 2
y = df_filtered['shares_difference']

data = pd.DataFrame({'shares_difference': y, 'averageUserRating': X1, 'in_app_dummy': X2, 'price_in_EUR' : X, 'apps_in_genre': Z})
data['const'] = 1


model = IV2SLS(
    dependent=data['shares_difference'],                  
    exog=data[['const', 'averageUserRating', 'in_app_dummy']],  
    instruments=data['apps_in_genre'],            
    endog=data['price_in_EUR']       
)

#  model fitting
results = model.fit()
print(results.summary) 

# the first stage results
first_stage_results = results.first_stage
print(first_stage_results.summary)


                          IV-2SLS Estimation Summary                          
Dep. Variable:      shares_difference   R-squared:                      0.0070
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0005
No. Observations:                 457   F-statistic:                    7.8718
Date:                Sun, Nov 10 2024   P-value (F-stat)                0.0487
Time:                        22:48:51   Distribution:                  chi2(3)
Cov. Estimator:                robust                                         
                                                                              
                                 Parameter Estimates                                 
                   Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-------------------------------------------------------------------------------------
const                -7.4883     1.0391    -7.2064     0.0000     -9.5249     -5.4516
averageUserRating     0.

In [15]:
df['market_share'] = df['downloads'] / total_downloads

alpha = 0.0318

# Calculate price elasticity of demand using the standard formula
df['derivative'] = -alpha * df['price_in_EUR'] * (1 - df['market_share'])

# Calculate markup using elasticity
df['markup'] = df['market_share'] / (-df['derivative'])

# Calculate marginal cost
df['marginal_cost'] = df['price_in_EUR'] - df['markup']

In [16]:
df[['marginal_cost', 'markup']]

,marginal_cost,markup
0,7.989987,0.000013
1,6.989975,0.000025
2,9.170866,0.000134
3,0.261955,0.001055
4,8.208027,0.000330
...,...,...
5495,2.989997,0.000003
5496,9.170449,0.000551
5497,0.554189,0.057211
5498,6.725370,0.000030
